In [ ]:
pip install torch transformers datasets scikit-learn accelerate


In [ ]:
from google.colab import drive


In [ ]:
drive.mount('/content/gdrive',force_remount=True)
%cd '/content/gdrive/MyDrive/Python_Thu5/comment_classification'

Mounted at /content/gdrive
/content/gdrive/MyDrive/Python_Thu5/comment_classification


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/binh_luan_sentiment_10000.csv')

In [ ]:
print(df.duplicated(subset=["Bình luận", "Sentiment Label"]).sum())


0


In [ ]:
label_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2,
    "Spam": 3
}

df = df.rename(columns={
    "Bình luận": "text",
    "Sentiment Label": "label"
})

df["label"] = df["label"].map(label_map)

df.to_csv("bds_clean.csv", index=False)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

train_df.to_csv("bds_train.csv", index=False)
test_df.to_csv("bds_test.csv", index=False)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files={
        "train": "bds_train.csv",
        "test": "bds_test.csv"
    }
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
)

# BẮT BUỘC cho PhoBERT
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize_function, batched=True)
dataset = dataset.rename_column("label", "labels")

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


Map:   0%|          | 0/1367 [00:00<?, ? examples/s]

Map:   0%|          | 0/342 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average=None,
        zero_division=0
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1_negative": f1[0],
        "f1_neutral": f1[1],
        "f1_positive": f1[2],
        "f1_spam": f1[3],
    }


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bds_comment_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,   # 👈 giảm từ 16
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none"
)


In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator   # 👈 QUAN TRỌNG
)

NameError: name 'tokenizer' is not defined

In [ ]:
MODEL_PATH = "/content/gdrive/MyDrive/Python_Thu5/comment_classification/bds_comment_model"

# ÉP lưu weight .bin
model.save_pretrained(
    MODEL_PATH,
    safe_serialization=False
)

# Lưu tokenizer
tokenizer.save_pretrained(MODEL_PATH)

# Kiểm tra file
import os
print("Files saved:")
for f in os.listdir(MODEL_PATH):
    print(" -", f)


Files saved:
 - checkpoint-47
 - checkpoint-94
 - checkpoint-141
 - config.json
 - tokenizer_config.json
 - special_tokens_map.json
 - sentencepiece.bpe.model
 - tokenizer.json
 - training_args.bin
 - pytorch_model.bin
 - added_tokens.json
 - vocab.txt
 - bpe.codes


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(dataset["test"])

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Negative", "Neutral", "Positive", "Spam"]
))


Confusion Matrix:
[[27  8 19 36]
 [59  0  8 17]
 [30 10 18 33]
 [36  7  5 29]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.18      0.30      0.22        90
     Neutral       0.00      0.00      0.00        84
    Positive       0.36      0.20      0.26        91
        Spam       0.25      0.38      0.30        77

    accuracy                           0.22       342
   macro avg       0.20      0.22      0.20       342
weighted avg       0.20      0.22      0.19       342



In [ ]:
model.save_pretrained("./bds_comment_model")
tokenizer.save_pretrained("./bds_comment_model")

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:461: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'max_length': 256}
  warnings.warn(


('./bds_comment_model/tokenizer_config.json',
 './bds_comment_model/special_tokens_map.json',
 './bds_comment_model/sentencepiece.bpe.model',
 './bds_comment_model/added_tokens.json',
 './bds_comment_model/tokenizer.json')

In [ ]:
import torch

LABELS = ["Negative", "Neutral", "Positive", "Spam"]

def classify_comment(text: str):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    label_id = torch.argmax(logits, dim=1).item()
    return LABELS[label_id]


In [ ]:
print(classify_comment("Dự đoán hợp lý để tham khảo trước khi mua bán."))


Positive


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(dataset["test"])

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Negative", "Neutral", "Positive", "Spam"]
))


NameError: name 'trainer' is not defined

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

LABELS = ["Negative", "Neutral", "Positive", "Spam"]

def classify_comment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    label_id = torch.argmax(outputs.logits, dim=1).item()
    return LABELS[label_id]

print(classify_comment("ha haha ."))


Spam
